Cell 1

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import clone
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

Cell 2

In [ ]:
# rebuild feature_df and groups
feature_df = pd.read_parquet("data/processed/feature_df.parquet").copy()
groups = feature_df["user"].astype(str)
y_all = feature_df["is_insider"].astype(int)

leak_cols = [
    "after_hours_events",
    "user_mean_after",
    "user_std_after",
    "after_hours_events_dev",
    "after_hours_ratio",
]
mean_std_cols = [c for c in feature_df.columns if c.startswith("user_mean_") or c.startswith("user_std_")]
drop_cols = list(set(leak_cols + mean_std_cols + ["user", "day", "employee_name", "email", "projects", "total_events"]))

X_all = feature_df.drop(columns=[c for c in drop_cols if c in feature_df.columns]).copy()
X_all = X_all.drop(columns=["is_insider"])

cat_cols = [c for c in ["role", "functional_unit", "department", "team", "supervisor"] if c in X_all.columns]
X_all = pd.get_dummies(X_all, columns=cat_cols, drop_first=False)

print("X_all:", X_all.shape)
print("Positives:", int(y_all.sum()))

Cell 3

In [ ]:
def eval_probs(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    return {
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "pr_auc": average_precision_score(y_true, y_proba),
    }

Cell 4

In [ ]:
def cross_val_model(name, model, X, y, groups, n_splits=5):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_rows = []
    
    for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
        
        m = clone(model)
        m.fit(X_tr, y_tr)
        proba = m.predict_proba(X_te)[:, 1]
        
        scores = eval_probs(y_te, proba, threshold=0.5)
        scores["model"] = name
        scores["fold"] = fold
        scores["positives_test"] = int(y_te.sum())
        scores["negatives_test"] = int((y_te == 0).sum())
        fold_rows.append(scores)
        
    return pd.DataFrame(fold_rows)

Cell 5

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


lr_model = LogisticRegression(class_weight="balanced", max_iter=5000)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    min_child_weight=1,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    scale_pos_weight=10
)

cv_lr = cross_val_model("Logistic Regression", lr_model, X_all, y_all, groups, n_splits=5)
cv_rf = cross_val_model("Random Forest", rf_model, X_all, y_all, groups, n_splits=5)
cv_xgb = cross_val_model("XGBoost", xgb_model, X_all, y_all, groups, n_splits=5)

cv_results = pd.concat([cv_lr, cv_rf, cv_xgb], ignore_index=True)
cv_results

Cell 6

In [ ]:
summary = cv_results.groupby("model").agg(
    folds=("fold", "count"),
    precision_mean=("precision", "mean"),
    precision_std=("precision", "std"),
    recall_mean=("recall", "mean"),
    recall_std=("recall", "std"),
    f1_mean=("f1", "mean"),
    f1_std=("f1", "std"),
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_std=("roc_auc", "std"),
    pr_auc_mean=("pr_auc", "mean"),
    pr_auc_std=("pr_auc", "std"),
    positives_test_mean=("positives_test", "mean"),
).reset_index()

summary

Cell 7

In [ ]:
cv_results.to_csv("data/processed/cv_fold_results.csv", index=False)
summary.to_csv("data/processed/cv_summary.csv", index=False)

In [ ]:
import joblib

joblib.dump(xgb_model, "data/models/xgb_cv.joblib")